In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
data = '../data/prices.csv'

In [3]:
df = pd.read_csv(data)

In [4]:
df.head().T

,0,1,2,3,4
date,2008-01-15,2008-01-15,2008-01-15,2008-01-15,2008-01-15
admin1,Kachin,Kachin,Kachin,Kachin,Kachin
admin2,Myitkyina,Myitkyina,Myitkyina,Myitkyina,Myitkyina
market,Wai Maw,Wai Maw,Wai Maw,Wai Maw,Wai Maw
market_id,1021,1021,1021,1021,1021
latitude,25.34,25.34,25.34,25.34,25.34
longitude,97.44,97.44,97.44,97.44,97.44
category,cereals and tubers,"meat, fish and eggs","meat, fish and eggs",miscellaneous food,vegetables and fruits
commodity,Rice (low quality),Meat (chicken),Meat (pork),Salt,Onions
commodity_id,145,94,140,185,173


In [5]:
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 61790 entries, 0 to 61789
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          61790 non-null  str    
 1   admin1        61790 non-null  str    
 2   admin2        61790 non-null  str    
 3   market        61790 non-null  str    
 4   market_id     61790 non-null  int64  
 5   latitude      61790 non-null  float64
 6   longitude     61790 non-null  float64
 7   category      61790 non-null  str    
 8   commodity     61790 non-null  str    
 9   commodity_id  61790 non-null  int64  
 10  unit          61790 non-null  str    
 11  priceflag     61790 non-null  str    
 12  pricetype     61790 non-null  str    
 13  currency      61790 non-null  str    
 14  price         61790 non-null  float64
 15  usdprice      61790 non-null  float64
dtypes: float64(4), int64(2), str(10)
memory usage: 7.5 MB
None


In [6]:
df['date'] = pd.to_datetime(df['date'])

## Turn Everything into small letters

In [7]:
df.columns = df.columns.str.lower().str.replace(' ', '_')


for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].str.lower().str.replace(' ', '_')


In [8]:
unique_commodities = df['commodity'].unique()
print(f"Total unique commodities: {unique_commodities}")

commodity_dfs = {}

for comm in unique_commodities:
    commodity_dfs[comm] = df[df['commodity'] == comm].copy()

unique_markets = df['market'].unique()
print(f"Total unique market: {unique_markets}")

Total unique commodities: <StringArray>
[                           'rice_(low_quality)',
                                'meat_(chicken)',
                                   'meat_(pork)',
                                          'salt',
                                        'onions',
                                      'tomatoes',
                                      'potatoes',
                                    'oil_(palm)',
                             'chickpeas_(local)',
                                         'maize',
                           'rice_(high_quality)',
                                      'soybeans',
                                        'garlic',
                                   'meat_(beef)',
                               'oil_(groundnut)',
                         'oil_(mixed,_imported)',
                                  'rice_(emata)',
                                     'chickpeas',
                                        'pulses',
          

Total unique market: <StringArray>
[       'wai_maw',         'magway',    'yenanchaung',      'pakhangyi',
 'alel_than_kyaw',     'buthidaung',    'kyein_chung',       'maungdaw',
      'myin_hlut',   'nyaung_chung',
 ...
    'nyaunglebin',      'htantabin',         'monywa',       'shwegyin',
       'kyaukkyi',         'hpapun',      'ponnagyun',     'taung_shey',
  'aungpan_myoma',    'yoe_ta_yoke']
Length: 300, dtype: str


In [9]:
# Create a dictionary where keys are (commodity, market) tuples
comm_market_dfs = {}

for comm in unique_commodities:
    for mkt in unique_markets:
        # Filter for the specific commodity and market
        subset = df[(df['commodity'] == comm) & (df['market'] == mkt)]
        
        # Only save it if data actually exists for this combination
        if not subset.empty:
            comm_market_dfs[(comm, mkt)] = subset.copy()



In [10]:
commodity_market_dates = (
    df.groupby(['commodity', 'market'])['date']
      .agg(min_date='min', max_date='max')
      .reset_index()
      .sort_values(['commodity', 'market'])
)

display(commodity_market_dates) 

,commodity,market,min_date,max_date
0,chickpeas,yangon,2011-01-15,2020-02-15
1,chickpeas_(local),ah_nauk_pyin,2020-11-15,2020-11-15
2,chickpeas_(local),ah_nauk_ywe,2020-11-15,2020-11-15
3,chickpeas_(local),ahpauk_wa,2020-11-15,2020-11-15
4,chickpeas_(local),alel_than_kyaw,2008-01-15,2011-01-15
...,...,...,...,...
2500,"wage_(non-qualified_labour,_non-agricultural)",wai_maw,2014-01-15,2020-03-15
2501,"wage_(non-qualified_labour,_non-agricultural)",wang_leng,2014-01-15,2017-04-15
2502,"wage_(non-qualified_labour,_non-agricultural)",yaesagyo,2014-01-15,2017-12-15
2503,"wage_(non-qualified_labour,_non-agricultural)",yenanchaung,2014-01-15,2016-08-15


In [11]:
commodity_market_dates_pivot = commodity_market_dates.pivot(
    index='commodity',
    columns='market',
    values=['min_date', 'max_date']
)



In [12]:
max_dates = df.groupby(['commodity', 'market'])['date'].max().rename('max_date')

pairs_max_date_2026 = (
    max_dates[max_dates.dt.year == 2026]
    .reset_index()
    .sort_values(['commodity', 'market'])
)
    
display(pairs_max_date_2026)

,commodity,market,max_date
0,eggs_(local),aungpan_myoma,2026-09-15
1,eggs_(local),baw_du_pha,2026-06-15
2,eggs_(local),bhamo,2026-09-15
3,eggs_(local),bhamo_market,2026-08-15
4,eggs_(local),bi_kin,2026-04-15
...,...,...,...
867,tomatoes_(local),ywarma,2026-09-15
868,tomatoes_(local),zami_yadana,2026-09-15
869,tomatoes_(local),zamyinzwel,2026-09-15
870,tomatoes_(local),zay_cho,2026-06-15


In [13]:
max_dates.index.get_level_values('commodity').nunique()

25

In [14]:
coverage = (
    df.groupby(['commodity', 'market'])
    .agg(
        first_date=('date', 'min'),
        last_date=('date', 'max'),
        observations=('date', 'count'),
        unique_months=('date', lambda x: x.dt.to_period('M').nunique())
    )
    .reset_index()
)

coverage = coverage.sort_values(
    'unique_months',
    ascending=False
)

display(coverage.head(20))

,commodity,market,first_date,last_date,observations,unique_months
1694,rice_(low_quality),lashio_market,2008-01-15,2026-09-15,202,202
1990,salt,lashio_market,2008-04-15,2026-09-15,179,179
690,oil_(palm),buthidaung,2008-01-15,2026-09-15,168,168
1616,rice_(low_quality),buthidaung,2008-01-15,2026-08-15,162,162
1788,rice_(low_quality),pakokku_myo_ma_market,2014-01-15,2026-09-15,146,146
2082,salt,pakokku_myo_ma_market,2014-01-15,2026-09-15,146,146
1634,rice_(low_quality),hakha_myoma_market,2010-06-15,2025-03-15,139,139
1930,salt,hakha_myoma_market,2010-06-15,2025-02-15,139,139
1912,salt,buthidaung,2011-02-15,2026-09-15,138,138
1708,rice_(low_quality),magway,2008-01-15,2020-03-15,135,135


In [15]:
coverage['unique_months'].describe()

count    2505.000000
mean       24.666667
std        25.663719
min         1.000000
25%         5.000000
50%        14.000000
75%        40.000000
max       202.000000
Name: unique_months, dtype: float64

In [16]:
coverage_summary = (
    coverage.groupby("commodity")["unique_months"]
    .agg(["count", "median", "max"])
    .sort_values("max", ascending=False)
)

display(coverage_summary)

,count,median,max
commodity,,,
rice_(low_quality),298,23.5,202
salt,294,24.0,179
oil_(palm),274,18.0,168
oil_(groundnut),39,1.0,135
pulses,276,18.0,124
chickpeas_(local),124,1.0,101
rice_(emata),1,97.0,97
chickpeas,1,79.0,79
eggs_(local),250,17.0,74


In [17]:
# How many combinations meet each minimum coverage threshold?
for threshold in [12, 24, 36, 60, 100]:
    count = (coverage["unique_months"] >= threshold).sum()
    print(f"{threshold}+ months: {count} combinations")

12+ months: 1387 combinations
24+ months: 1007 combinations
36+ months: 743 combinations
60+ months: 241 combinations
100+ months: 47 combinations


In [19]:
# Commodity-market pairs whose latest observation is in 2026
# and that have at least 60 unique months of data
df_2026_60plus = (
    pairs_max_date_2026
    .merge(coverage, on=['commodity', 'market'], how='inner')
    .query('unique_months >= 36')
    .sort_values(['commodity', 'market'])
    .reset_index(drop=True)
)

display(df_2026_60plus)

,commodity,market,max_date,first_date,last_date,observations,unique_months
0,eggs_(local),bilin_myo_ma_market,2026-05-15,2020-07-15,2026-05-15,46,46
1,eggs_(local),butar,2026-07-15,2022-09-15,2026-07-15,38,38
2,eggs_(local),buthidaung,2026-09-15,2020-07-15,2026-09-15,64,64
3,eggs_(local),hlaing_bwe_myo_ma_market,2026-09-15,2020-07-15,2026-09-15,44,44
4,eggs_(local),jan_mai_kaung,2026-09-15,2022-09-15,2026-09-15,45,45
...,...,...,...,...,...,...,...
357,tomatoes_(local),yae_baw_market,2026-09-15,2020-07-15,2026-09-15,42,42
358,tomatoes_(local),yaenimyaung,2026-09-15,2021-07-15,2026-09-15,45,45
359,tomatoes_(local),yebyu_market,2026-09-15,2020-07-15,2026-09-15,47,47
360,tomatoes_(local),ywarma,2026-09-15,2021-07-15,2026-09-15,59,59


In [20]:
df_2026_60plus["unique_months"].describe()

count    362.000000
mean      59.436464
std       22.869680
min       36.000000
25%       45.250000
50%       56.000000
75%       65.000000
max      202.000000
Name: unique_months, dtype: float64

In [21]:
display(
    df_2026_60plus.sort_values(
        "unique_months",
        ascending=False
    ).head(20)
)

,commodity,market,max_date,first_date,last_date,observations,unique_months
223,rice_(low_quality),lashio_market,2026-09-15,2008-01-15,2026-09-15,202,202
281,salt,lashio_market,2026-09-15,2008-04-15,2026-09-15,179,179
55,oil_(palm),buthidaung,2026-09-15,2008-01-15,2026-09-15,168,168
210,rice_(low_quality),buthidaung,2026-08-15,2008-01-15,2026-08-15,162,162
243,rice_(low_quality),pakokku_myo_ma_market,2026-09-15,2014-01-15,2026-09-15,146,146
301,salt,pakokku_myo_ma_market,2026-09-15,2014-01-15,2026-09-15,146,146
267,salt,buthidaung,2026-09-15,2011-02-15,2026-09-15,138,138
276,salt,kutkai,2026-09-15,2014-01-15,2026-09-15,130,130
218,rice_(low_quality),kutkai,2026-09-15,2014-01-15,2026-09-15,130,130
309,salt,taunggyi,2026-09-15,2008-01-15,2026-09-15,127,127


In [22]:
candidate_keys = df_2026_60plus[
    ["commodity", "market"]
].drop_duplicates()

candidate_details = df.merge(
    candidate_keys,
    on=["commodity", "market"],
    how="inner"
)

display(
    candidate_details.groupby(
        ["commodity", "market", "unit", "pricetype", "currency"]
    ).size().reset_index(name="records")
)

,commodity,market,unit,pricetype,currency,records
0,eggs_(local),bilin_myo_ma_market,10_pcs,retail,mmk,46
1,eggs_(local),butar,10_pcs,retail,mmk,38
2,eggs_(local),buthidaung,10_pcs,retail,mmk,64
3,eggs_(local),hlaing_bwe_myo_ma_market,10_pcs,retail,mmk,44
4,eggs_(local),jan_mai_kaung,10_pcs,retail,mmk,45
...,...,...,...,...,...,...
357,tomatoes_(local),yae_baw_market,kg,retail,mmk,42
358,tomatoes_(local),yaenimyaung,kg,retail,mmk,45
359,tomatoes_(local),yebyu_market,kg,retail,mmk,47
360,tomatoes_(local),ywarma,kg,retail,mmk,59


In [23]:
candidate_summary = (
    df_2026_60plus
    .groupby("commodity")
    .agg(
        candidate_markets=("market", "nunique"),
        max_months=("unique_months", "max"),
        median_months=("unique_months", "median")
    )
    .sort_values("candidate_markets", ascending=False)
)

display(candidate_summary)

,candidate_markets,max_months,median_months
commodity,,,
salt,59,179,60.0
rice_(low_quality),57,202,60.0
pulses,53,122,60.0
oil_(palm),52,168,56.0
eggs_(local),51,74,50.0
onions_(local),50,74,53.5
tomatoes_(local),40,74,47.0


In [24]:
display(
    df_2026_60plus
    .sort_values("unique_months", ascending=False)
    .head(20)
)

,commodity,market,max_date,first_date,last_date,observations,unique_months
223,rice_(low_quality),lashio_market,2026-09-15,2008-01-15,2026-09-15,202,202
281,salt,lashio_market,2026-09-15,2008-04-15,2026-09-15,179,179
55,oil_(palm),buthidaung,2026-09-15,2008-01-15,2026-09-15,168,168
210,rice_(low_quality),buthidaung,2026-08-15,2008-01-15,2026-08-15,162,162
243,rice_(low_quality),pakokku_myo_ma_market,2026-09-15,2014-01-15,2026-09-15,146,146
301,salt,pakokku_myo_ma_market,2026-09-15,2014-01-15,2026-09-15,146,146
267,salt,buthidaung,2026-09-15,2011-02-15,2026-09-15,138,138
276,salt,kutkai,2026-09-15,2014-01-15,2026-09-15,130,130
218,rice_(low_quality),kutkai,2026-09-15,2014-01-15,2026-09-15,130,130
309,salt,taunggyi,2026-09-15,2008-01-15,2026-09-15,127,127
